## Import libs

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.python.data import Dataset
from tensorflow.keras.optimizers import Adam
import seaborn as sns


from falsb4mpa.modeling.zhang.learning.multi_adv import train_loop as zhang_train
from falsb4mpa.dataset.load_data import load_data
from falsb4mpa.evaluation.evaluation import compute_predictive_metrics, compute_fair_metrics, compute_adv_metrics, compute_tradeoff, fair_evaluation, compute_intersectional_fair_metrics
from falsb4mpa.modeling.zhang.models.multi_adv import ZhangMultAdv

## Preliminaries

In [2]:
batch_size = 64
epochs = 100
learning_rate = 0.001

In [3]:
cv_seeds = [55, 73]
# cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'adult-mpa-cat-wout-agg'

In [5]:
x, y, a1, a2 = load_data(data_name)
raw_data = (x, y, a1, a2)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
a1dim = a1.shape[1]
a2dim = a2.shape[1]
zdim = 8
print(xdim, ydim, a1dim, a2dim, zdim)

102 1 1 5 8


## Result file

In [7]:
header = [
    "model_name", "cv_seed", 
    "clas_acc", "f1-micro", "f1-macro",
    "a1_dp", "a1_deqodds", "a1_deqopp",
    "a2_dp", "a2_deqodds", "a2_deqopp",
    "wc_spd", "wc_aod", "wc_eod",
    "last_cosine_similarity"
]

results = []

## Testing

### DemPar

In [ ]:
fairdef = "DemPar"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvCat4DP', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] 
    result += [a2_dp, a2_deqodds, a2_deqopp]
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

### EqOdds

In [8]:
fairdef = "EqOdds"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdvCat4EqOdds', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] 
    result += [a2_dp, a2_deqodds, a2_deqopp]
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

2026-05-15 06:29:28.940921: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 1 | Clf loss/acc 0.30/0.73 | Adv1 loss/acc 0.73/0.67 | Adv2 loss/acc 1.34/0.86 | Cos Sim -0.03


2026-05-15 06:29:44.436982: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 2 | Clf loss/acc 0.26/0.83 | Adv1 loss/acc 0.79/0.67 | Adv2 loss/acc 1.14/0.86 | Cos Sim -0.16
> Epoch: 3 | Clf loss/acc 0.25/0.83 | Adv1 loss/acc 0.87/0.67 | Adv2 loss/acc 1.01/0.86 | Cos Sim -0.26


2026-05-15 06:30:15.669110: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


...

> Epoch: 32 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.24/0.67 | Adv2 loss/acc 1.03/0.86 | Cos Sim -0.16
> Epoch: 33 | Clf loss/acc 0.42/0.84 | Adv1 loss/acc 2.25/0.67 | Adv2 loss/acc 1.03/0.86 | Cos Sim -0.15
> Epoch: 34 | Clf loss/acc 0.43/0.84 | Adv1 loss/acc 2.27/0.67 | Adv2 loss/acc 1.04/0.86 | Cos Sim -0.15
> Epoch: 35 | Clf loss/acc 0.43/0.84 | Adv1 loss/acc 2.28/0.67 | Adv2 loss/acc 1.04/0.86 | Cos Sim -0.15
> Epoch: 36 | Clf loss/acc 0.44/0.84 | Adv1 loss/acc 2.29/0.67 | Adv2 loss/acc 1.04/0.86 | Cos Sim -0.14
> Epoch: 37 | Clf loss/acc 0.44/0.84 | Adv1 loss/acc 2.30/0.67 | Adv2 loss/acc 1.04/0.86 | Cos Sim -0.14
> Epoch: 38 | Clf loss/acc 0.44/0.84 | Adv1 loss/acc 2.31/0.67 | Adv2 loss/acc 1.04/0.86 | Cos Sim -0.14
> Epoch: 39 | Clf loss/acc 0.45/0.84 | Adv1 loss/acc 2.32/0.67 | Adv2 loss/acc 1.04/0.86 | Cos Sim -0.13
> Epoch: 40 | Clf loss/acc 0.45/0.84 | Adv1 loss/acc 2.33/0.67 | Adv2 loss/acc 1.04/0.86 | Cos Sim -0.13
> Epoch: 41 | Clf loss/acc 0.45/0.84 | Adv1 loss/acc 2.

2026-05-15 06:45:54.331715: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 64 | Clf loss/acc 0.48/0.84 | Adv1 loss/acc 2.46/0.67 | Adv2 loss/acc 1.05/0.86 | Cos Sim -0.09
> Epoch: 65 | Clf loss/acc 0.49/0.84 | Adv1 loss/acc 2.47/0.67 | Adv2 loss/acc 1.05/0.86 | Cos Sim -0.09
> Epoch: 66 | Clf loss/acc 0.49/0.84 | Adv1 loss/acc 2.47/0.67 | Adv2 loss/acc 1.05/0.86 | Cos Sim -0.09
> Epoch: 67 | Clf loss/acc 0.49/0.84 | Adv1 loss/acc 2.47/0.67 | Adv2 loss/acc 1.05/0.86 | Cos Sim -0.09
> Epoch: 68 | Clf loss/acc 0.49/0.84 | Adv1 loss/acc 2.48/0.67 | Adv2 loss/acc 1.05/0.86 | Cos Sim -0.09
> Epoch: 69 | Clf loss/acc 0.49/0.84 | Adv1 loss/acc 2.48/0.67 | Adv2 loss/acc 1.05/0.86 | Cos Sim -0.09
> Epoch: 70 | Clf loss/acc 0.49/0.84 | Adv1 loss/acc 2.48/0.67 | Adv2 loss/acc 1.05/0.86 | Cos Sim -0.09
> Epoch: 71 | Clf loss/acc 0.49/0.84 | Adv1 loss/acc 2.48/0.67 | Adv2 loss/acc 1.05/0.86 | Cos Sim -0.09
> Epoch: 72 | Clf loss/acc 0.50/0.84 | Adv1 loss/acc 2.49/0.67 | Adv2 loss/acc 1.05/0.86 | Cos Sim -0.09
> Epoch: 73 | Clf loss/acc 0.50/0.84 | Adv1 loss/acc 2.

2026-05-15 07:02:46.916695: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> Epoch: 27 | Clf loss/acc 0.53/0.84 | Adv1 loss/acc 1.70/0.67 | Adv2 loss/acc 0.95/0.86 | Cos Sim -0.22
> Epoch: 28 | Clf loss/acc 0.53/0.84 | Adv1 loss/acc 1.71/0.67 | Adv2 loss/acc 0.95/0.86 | Cos Sim -0.22
> Epoch: 29 | Clf loss/acc 0.54/0.84 | Adv1 loss/acc 1.73/0.67 | Adv2 loss/acc 0.96/0.86 | Cos Sim -0.22
> Epoch: 30 | Clf loss/acc 0.54/0.84 | Adv1 loss/acc 1.74/0.67 | Adv2 loss/acc 0.96/0.86 | Cos Sim -0.22
> Epoch: 31 | Clf loss/acc 0.54/0.84 | Adv1 loss/acc 1.75/0.67 | Adv2 loss/acc 0.96/0.86 | Cos Sim -0.22
> Epoch: 32 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.76/0.67 | Adv2 loss/acc 0.96/0.86 | Cos Sim -0.22
> Epoch: 33 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.77/0.67 | Adv2 loss/acc 0.96/0.86 | Cos Sim -0.22
> Epoch: 34 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.78/0.67 | Adv2 loss/acc 0.97/0.86 | Cos Sim -0.22
> Epoch: 35 | Clf loss/acc 0.55/0.84 | Adv1 loss/acc 1.79/0.67 | Adv2 loss/acc 0.97/0.86 | Cos Sim -0.22
> Epoch: 36 | Clf loss/acc 0.56/0.84 | Adv1 loss/acc 1.

### EqOpp

In [ ]:
fairdef = "EqOpp"

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test = train_test_split(
        x, y, a1, a2, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a1_train, a2_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a1_test, a2_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    opt = Adam(learning_rate=learning_rate)

    model = ZhangMultAdv(xdim=xdim, ydim=ydim, a1dim=a1dim, a2dim=a2dim, batch_size=batch_size, fairdef=fairdef)
    
    ret, dULa1, dULa2, cos_sim = zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A1, A2, Y_hat, A1_hat, A2_hat = fair_evaluation(model, test_data)
    
    clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix = compute_predictive_metrics(Y, Y_hat)
    
    adv1_acc = compute_adv_metrics(A1, A1_hat)
    adv2_acc = compute_adv_metrics(A2, A2_hat)
    
    a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1 = compute_fair_metrics(Y, A1, Y_hat, a1dim)
    a2_dp, a2_deqodds, a2_deqopp = compute_fair_metrics(Y, A2, Y_hat, a2dim)

    wc_spd, wc_aod, wc_eod = compute_intersectional_fair_metrics(Y, A1, A2, Y_hat, a1dim, a2dim)


    # fair_metrics = (dp, deqodds, deqopp)
    # tradeoff = []
    # for fair_metric in fair_metrics:
    #     tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    # result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    result = ['MultAdv4EqOpp', cv_seed]
    result += [clas_acc, clas_f1_micro, clas_f1_macro]
    result += [a1_dp, a1_deqodds, a1_deqopp] 
    result += [a2_dp, a2_deqodds, a2_deqopp]
    result += [wc_spd, wc_aod, wc_eod]
    result += [cos_sim]


    results.append(result)

    del(opt, x_train, x_test, y_train, y_test, a1_train, a1_test, a2_train, a2_test, train_data, test_data, model, ret)
    del(Y, A1, A2, Y_hat, A1_hat, A2_hat)
    del(clas_acc, clas_f1_micro, clas_f1_macro, confusion_matrix, adv1_acc, adv2_acc)
    del(a1_dp, a1_deqodds, a1_deqopp, a1_metrics_g0, a1_metrics_g1, a2_dp, a2_deqodds, a2_deqopp)
    del(wc_spd, wc_aod, wc_eod)
    del(cos_sim)

## Saving into DF then CSV

In [9]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,f1-micro,f1-macro,a1_dp,a1_deqodds,a1_deqopp,a2_dp,a2_deqodds,a2_deqopp,wc_spd,wc_aod,wc_eod,last_cosine_similarity
0,MultAdvCat4EqOdds,55,0.832346,0.832346,0.769726,0.798907,0.894804,0.888158,0.226292,0.349176,0.226292,0.688938,0.601420,0.347170,-0.091718
1,MultAdvCat4EqOdds,73,0.832198,0.832198,0.772442,0.784549,0.887321,0.882097,0.243252,0.377413,0.243252,0.672397,0.586622,0.329744,-0.171314


In [10]:
result_df.to_csv(f'../../results/{data_name}-mult_adv-{epochs}.csv')